## 1. DETR (DEtection TRansformer)

**Contents:**
1. What & Why
2. Architecture
3. Object Queries
4. Set Prediction & Bipartite Hungarian Matching
5. Loss
6. Usage (Hugging Face)
7. References

---

**DETR** (Carion et al., ECCV 2020, Facebook AI) reframes object detection as a **direct set prediction** problem solved with a **Transformer encoder-decoder**. Its headline feature is that it removes almost all of the hand-designed components that classic detectors rely on:

- **no anchors / default boxes**,
- **no region proposals**,
- **no Non-Maximum Suppression (NMS)**.

Instead the model directly outputs a fixed-size **set** of predictions, and a **bipartite matching** loss forces each ground-truth object to be explained by exactly one prediction. This makes the pipeline genuinely **end-to-end** and conceptually clean, at the cost of slow convergence (the original DETR needs very long training schedules; follow-ups like Deformable DETR and DINO fix this).

## 2. Architecture

```
Image
  |
  v
CNN Backbone (e.g. ResNet-50)  ->  feature map (C x H x W)
  |  (1x1 conv to d dims, + fixed/learned positional encodings)
  v
Flatten to a sequence of HxW tokens
  |
  v
Transformer ENCODER  (self-attention over all image tokens)
  |
  v
Transformer DECODER
   inputs: N learned "object queries"  +  encoder memory
   (self-attention among queries, cross-attention to image)
  |
  v
N output embeddings
  |
  +--> FFN -> class label (incl. "no object" / empty)
  +--> FFN -> box (cx, cy, w, h), normalized to [0, 1]
```

The **encoder** uses global self-attention so every image location can attend to every other — useful for reasoning about object relations and disambiguating duplicates. The **decoder** transforms `N` object queries into `N` output embeddings in parallel, each decoded into one (class, box) prediction by shared feed-forward heads.

## 3. Object Queries

**Object queries** are `N` (e.g. 100) **learned embedding vectors** — they are model parameters, learned during training, not derived from the image. They act as "slots": each query is the decoder's request "is there an object here / of this kind?", and the decoder fills each slot with at most one detection.

Because `N` is fixed and is chosen to be larger than the typical number of objects in an image, most queries learn to predict the special **"no object" (empty)** class. Different queries specialize to different regions/sizes of the image, effectively learning a soft, data-driven equivalent of anchors. Since each query produces exactly one box and the loss forbids two queries from claiming the same object, **no NMS is needed**.

## 4. Set Prediction & Bipartite Hungarian Matching

DETR always emits exactly `N` predictions. Training requires deciding **which prediction is responsible for which ground-truth object** — a one-to-one assignment. DETR solves this with the **Hungarian algorithm** to find the optimal **bipartite matching** between the `N` predictions and the (padded-to-`N`) set of ground truths.

The matching minimizes a cost combining how well a prediction's class and box agree with a ground truth:

```
cost(pred_i, gt_j) = -p_i(class_j)                     # class agreement
                     + L1(box_i, box_j)                # center/size distance
                     + (1 - GIoU(box_i, box_j))        # overlap quality
```

The Hungarian algorithm returns a permutation that pairs each ground-truth object with exactly one prediction (the rest are matched to "no object"). This **one-to-one** assignment is the crucial difference from anchor-based detectors, which assign many anchors to each object and then need NMS to collapse the duplicates.

## 5. Loss

Once the optimal matching is fixed, the training loss is computed over those pairs:

```
L = L_class + L_box
  L_class = cross-entropy over (C + 1) classes  (the +1 is the "no object" class)
  L_box   = lambda_L1 * ||b_pred - b_gt||_1  +  lambda_giou * (1 - GIoU(b_pred, b_gt))
```

Notes:

- The box loss combines **L1** (good for absolute position/size) with **GIoU** (scale-invariant, directly optimizes overlap).
- Unmatched predictions are supervised only by the "no object" class term, and that term is **down-weighted** to counter the imbalance (most of the `N` slots are empty).
- **Auxiliary losses** are added after every decoder layer (not just the last) to speed up and stabilize convergence.

## 6. Usage (Hugging Face)

DETR is most easily used through Hugging Face Transformers (`facebook/detr-resnet-50`), which provides both the model and its image processor.

```python
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection
from PIL import Image

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
model.eval()

image = Image.open("image.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Convert raw outputs to [x1, y1, x2, y2] boxes in image coordinates
target_sizes = torch.tensor([image.size[::-1]])   # (height, width)
results = processor.post_process_object_detection(
    outputs, target_sizes=target_sizes, threshold=0.9
)[0]

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    print(model.config.id2label[label.item()], round(score.item(), 3), box.tolist())
```

Note there is **no NMS step** in this pipeline — post-processing only rescales boxes and thresholds by score. (torchvision does not ship a DETR detector; the Transformers library is the standard route. A panoptic-segmentation variant, `DetrForSegmentation`, also exists.)

## 7. References

- Carion, Massa, Synnaeve, Usunier, Kirillov, Zagoruyko, *End-to-End Object Detection with Transformers*, ECCV 2020 — https://arxiv.org/abs/2005.12872
- Official code (Facebook Research) — https://github.com/facebookresearch/detr
- Hugging Face DETR docs — https://huggingface.co/docs/transformers/model_doc/detr
- Zhu et al., *Deformable DETR* (faster convergence) — https://arxiv.org/abs/2010.04159